Model of cooling cup


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

%matplotlib inline

In [3]:
def readfile(filename):
        # read the CSV file into a dataframe (i.e. like a Python spreadsheet)
    df = pd.read_csv(filename)
    
    # convert dataframe into a 2D Numpy array
    a = df.to_numpy()
    
    # the first column are the counters
    counter_array = a[:,0]
    
    # the second column are the time stamps
    time_array = a[:,1]
    
    # the second column are the time stamps
    # arduino_time_stamp_array = a[:,2]
    
    # the third column are your measurements: either pulseTime or distance depending on how you modified the code above
    temperature_array = a[:,2]
    rh_array = a[:,3]

    return (time_array, temperature_array, rh_array)
    

In [33]:


# https://illingcompany.com/product/12oz-standard-355ml-202x211-brite-can/?srsltid=AfmBOorGi1H67og6nzUoUixZYzyuDW54VpLvHtOzYKGDkJxL2sChPYh_

# Rough dimensions of can
pi = 3.142
r = 66.25 / 2  # eff radius
h = 122.25  # height

# estimating exposed surface based on cylinder

A = (2 * (pi * (r **2) )) + (h * (2 * pi * r))   # exposed surface
V = 355 * (1/(100^3)) # volume
rhoair = 1.2  # approx kg/m^3  ROUGH (depends on T)

# https://kg-m3.com/material/aluminum

rhoAl = 2712 # kg/m3  # density Al

# mass of the air is equal to desnity of air * volume of air
mair = rhoair * V # kg air
mAl = 13 * (1/1000) # NOAH REMEMBER TO WEIGH THE CAN TO GET THIS PREISE # mass of can kg
d = 0.095  # rough thickness of Al  

# https://www.engineeringtoolbox.com/thermal-conductivity-metals-d_858.html
k = 237 # conductivity of Al
print(mair,d)

# https://www.engineeringtoolbox.com/specific-heat-capacity-d_391.html
# go back to make sure this is correct
cair = 11005 # air J/kg/K
cAl = 897 # Al J/kg/K
heatcap = mAl*cAl + mair*cair

Tcel = 273.15 # to convert C to K
Tamb = 23.2 + Tcel # ambient temp (K)
T0   = 35.30 + Tcel   # starting temperature

sig = 5.67e-8  # Stefan-Boltzmann W/m^2/K^4
eps = 0.002    #  emissivity
kc = 0.002   # convection coefficient W/m^2/K

n = 0
dt = 0.1 # time step (s)
tn = 230
Tn = T0

t = []
T = []

while Tn > Tamb + 0.1:
    dHc = kc * A * (Tn-Tamb) * dt
    dHr = eps * sig * A * (Tn**4 - Tamb**4) * dt
    Tn = Tn - (dHc + dHr)/heatcap
    tn = tn + dt
    
    delt = (dHc + dHr)*d/k/A/dt  # temp drop across wall of can - check it is tiny
#    print(delt)
    t.append(tn)     # store the current time
    T.append(Tn-Tcel)     # store the current temperature
    # print(tn,Tn-Tcel)
    

plt.figure
plt.plot(t, T, label="Temperature (C)")
plt.xlabel("Time (s)")
plt.ylabel("Temperature (C)")
#    plt.ylim(0, 100)             
plt.legend()
plt.show()

4.135922330097087 0.095


In [21]:


%matplotlib qt


fig, ax = plt.subplots()


time_array, temperature_array, rh_array = readfile("/Users/sam/Desktop/Science One For Real/physics/SCIE001Physics/dataSorting/bestRuns/pretty good.csv")


color = 'tab:green'
ax.plot(t, T, marker='o', linestyle='--', color=color)


# plot temperature data "live" as it is recorded 
color = 'tab:blue'
ax.plot(time_array, temperature_array, marker='o', linestyle='--', color=color)
ax.set_ylabel("Temperature ($^\\circ C$)", color=color)
ax.set_ylim([15,50])
ax.grid(visible=True, axis='x')
ax.tick_params(axis='y', labelcolor=color)
ax.set_xlabel('time (s)')

# plot relative humidity data on the same plot but with its own y-axis
ax2 = ax.twinx()
color = 'tab:red'
ax2.plot(time_array, rh_array, marker='o', linestyle='--', color=color)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Relative Humidity (%)", color=color)
ax2.set_ylim([0,100])
ax2.grid(visible=True) 
ax2.tick_params(axis='y', labelcolor=color)           

fig.tight_layout()


In [73]:
def euler(eps, kc, initalTime, initalTemp): 
    
    # https://illingcompany.com/product/12oz-standard-355ml-202x211-brite-can/?srsltid=AfmBOorGi1H67og6nzUoUixZYzyuDW54VpLvHtOzYKGDkJxL2sChPYh_
    
    # Rough dimensions of can
    pi = 3.142
    r = 66.25 / 2  # eff radius
    h = 122.25  # height
    
    # estimating exposed surface based on cylinder
    
    A = (2 * (pi * (r **2) )) + (h * (2 * pi * r))   # exposed surface
    V = 355 * (1/(100^3)) # volume
    rhoair = 1.2  # approx kg/m^3  ROUGH (depends on T)
    
    # https://kg-m3.com/material/aluminum
    
    rhoAl = 2712 # kg/m3  # density Al
    
    # mass of the air is equal to desnity of air * volume of air
    mair = rhoair * V # kg air
    mAl = 13 * (1/1000) # NOAH REMEMBER TO WEIGH THE CAN TO GET THIS PREISE # mass of can kg
    d = 0.095  # rough thickness of Al  
    
    # https://www.engineeringtoolbox.com/thermal-conductivity-metals-d_858.html
    k = 237 # conductivity of Al
    print(mair,d)
    
    # https://www.engineeringtoolbox.com/specific-heat-capacity-d_391.html
    # go back to make sure this is correct
    cair = 11005 # air J/kg/K
    cAl = 897 # Al J/kg/K
    heatcap = mAl*cAl + mair*cair
    
    Tcel = 273.15 # to convert C to K
    Tamb = 23.2 + Tcel # ambient temp (K)
    T0   = initalTemp + Tcel   # starting temperature
    
    sig = 5.67e-8  # Stefan-Boltzmann W/m^2/K^4
    
    n = 0
    dt = 0.1 # time step (s)
    tn = initalTime
    Tn = T0
    
    t = []
    T = []

    t = []
    T = []

    while Tn > Tamb + 0.1:
        dHc = kc * A * (Tn-Tamb) * dt
        dHr = eps * sig * A * (Tn**4 - Tamb**4) * dt
        Tn = Tn - (dHc + dHr)/heatcap
        tn = tn + dt
        
        delt = (dHc + dHr)*d/k/A/dt  # temp drop across wall of can - check it is tiny
    #    print(delt)
        t.append(tn)     # store the current time
        T.append(Tn-Tcel)     # store the current temperature
        # print(tn,Tn-Tcel)

    return (t, T)

In [75]:
# https://matplotlib.org/stable/gallery/widgets/slider_demo.html

import matplotlib.pyplot as plt
import numpy as np

from matplotlib.widgets import Button, Slider

# The parametrized function to be plotted
# def f(t, amplitude, frequency):
#     return amplitude * np.sin(2 * np.pi * frequency * t)

time_array, temperature_array, rh_array = readfile("/Users/sam/Desktop/Science One For Real/physics/SCIE001Physics/dataSorting/bestRuns/pretty good.csv")

t = time_array

# Define initial parameters
init_eps = 0.002
init_kc = 0.001

init_temp = 35
init_time = 230

# Create the figure and the line that we will manipulate
fig, ax = plt.subplots()
t, T = euler(init_kc, init_eps, init_time, init_temp)
line, = ax.plot(t, T, lw=2, color='tab:red')
# ax.set_xlabel('Time [s]')

ax.plot(time_array, temperature_array, marker='o', linestyle='--', color='tab:blue', ms=2)
ax.set_ylabel("Temperature ($^\\circ C$)", color=color)
ax.set_ylim([15,50])
ax.grid(visible=True, axis='x')
ax.tick_params(axis='y', labelcolor=color)
ax.set_xlabel('time (s)')

# adjust the main plot to make room for the sliders
fig.subplots_adjust(left=0.25, bottom=0.25)

# Make a horizontal slider to control the frequency.
axfreq = fig.add_axes([0.25, 0.1, 0.65, 0.03])
kc_slider = Slider(
    ax=axfreq,
    label='kc',
    valmin=0.0,
    valmax=0.002,
    valinit=init_kc,
)

# Make a vertically oriented slider to control the amplitude
axamp = fig.add_axes([0.1, 0.25, 0.0225, 0.63])
eps_slider = Slider(
    ax=axamp,
    label="eps",
    valmin=0.0,
    valmax=0.02,
    valinit=init_eps,
    orientation="vertical"
)

# Make a horizontal slider to control the frequency.
axTimeStart = fig.add_axes([0.25, 0.05, 0.65, 0.03])
timeStart_slider = Slider(
    ax=axTimeStart,
    label='timeStart',
    valmin=min(time_array),
    valmax=max(time_array),
    valinit=init_time,
)

# Make a horizontal slider to control the frequency.
axTempStart = fig.add_axes([0.05, 0.25,  0.03, 0.65])
tempStart_slider = Slider(
    ax=axTempStart,
    label='tempStart',
    valmin=min(temperature_array),
    valmax=max(temperature_array),
    valinit=init_temp,
    orientation="vertical"
)


# The function to be called anytime a slider's value changes
def update(val):
    t, T = euler(kc_slider.val, eps_slider.val, timeStart_slider.val, tempStart_slider.val)
    line.set_ydata(T)
    line.set_xdata(t)
    fig.canvas.draw_idle()


# register the update function with each slider
kc_slider.on_changed(update)
eps_slider.on_changed(update)
timeStart_slider.on_changed(update)
tempStart_slider.on_changed(update)

# Create a `matplotlib.widgets.Button` to reset the sliders to initial values.
resetax = fig.add_axes([0.8, 0.025, 0.1, 0.04])
button = Button(resetax, 'Reset', hovercolor='0.975')


def reset(event):
    freq_slider.reset()
    amp_slider.reset()
    freq_slider.reset()
    amp_slider.reset()
    
button.on_clicked(reset)

plt.show()

4.135922330097087 0.095
